# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing a dataset using the `mlcroissant` library. All entities (record sets, fields, columns, etc.) are referenced by their `@id` fields to ensure reproducible and standards-based workflows.

### Dataset Source
The dataset is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset via Croissant schema
dataset = mlc.Dataset(croissant_url)

# Dataset metadata summary
metadata = dataset.metadata
print("{}: {}".format(metadata.name, metadata.description))
print("Version:", getattr(metadata, 'version', ''))
print("Published on:", getattr(metadata, 'datePublished', ''))

## 2. Data Overview
Review available record sets and their fields; all referenced by their `@id` fields.

Let's list all record sets, their `@id`, and the fields for each.

In [ ]:
# List available record sets and their fields
record_sets = dataset.metadata.recordSet

for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    print("  Fields:")
    for field in rs['field']:
        print(f"    - {field['@id']} ({field.get('name', '')})")
    print("  Columns:")
    for col in rs['column']:
        print(f"    - {col['@id']} ({col.get('name', '')})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll extract records for each record set by their `@id`. You can explore their columns and values.

In [ ]:
# Prepare to load data from each record set
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for {record_set_id}")

# Show columns for the first record set
first_record_set_id = record_set_ids[0]
print("Columns for record set", first_record_set_id, ":")
print(dataframes[first_record_set_id].columns.tolist())
dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps: filtering records by a numeric field, normalizing values, and categorizing/grouping attributes.

We'll pick key numeric fields such as GAD-7, PHQ-9, or PSQ for analysis (by their `@id`). All operations reference the corresponding `@id` fields from the schema.

In [ ]:
# Example: Filter, normalize, and group by demographic field

# Choose the record set for survey results
# (Replace with actual @id from your previous overview)
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# Choose the GAD-7 score field (fetch by @id from fields/columns overview in section 2)
numeric_field = 'cr:gad7_score'  # Example @id; update if needed

# Filter the DataFrame by a relevant threshold
threshold = 10
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by education level (example demographic field @id)
    group_field = 'cr:level_of_education'  # Example @id; update if needed
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[[numeric_field, f"{numeric_field}_normalized"]].mean()
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field} not found in record set {record_set_id}. Please update to a valid numeric @id from your schema.")

## 5. Visualization
Visualize distributions and relationships in the dataset. Reference fields by their `@id`.

Let's plot the normalized GAD-7 scores grouped by education level.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization: Distribution of normalized GAD-7 scores by education level
if numeric_field in df.columns and group_field in df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"Distribution of {numeric_field} scores by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()

# Histogram for normalized GAD-7
if f"{numeric_field}_normalized" in filtered_df.columns:
    plt.figure(figsize=(8, 4))
    filtered_df[f"{numeric_field}_normalized"].hist(bins=20)
    plt.title(f"Histogram of normalized {numeric_field} across filtered records")
    plt.xlabel(f"{numeric_field}_normalized")
    plt.ylabel("Count")
    plt.show()

## 6. Conclusion
This notebook demonstrated reproducible loading, exploration, and initial processing of AI-ready survey data using the Croissant standard and `mlcroissant`.

Key findings:
- The dataset offers a rich set of demographic and mental health fields identified by `@id` and standardized for interoperability.
- Data can be filtered and normalized for statistical analysis, grouped by demographic attributes for insight.
- Visualizations allow rapid exploration of relationships (e.g., GAD-7 distribution by education level).

For deeper analyses, reference all fields and record sets by their `@id` for full FAIR2 compliance.